# MovieLens 100K 영화 추천 데이터셋

약 1000명의 사용자가 약 1,700개 영화에 대해서 평가한 평점 10만개 데이터셋입니다.

[데이터 출처](https://grouplens.org/datasets/movielens/100k/)

In [1]:
import pandas as pd

In [ ]:
ratings = pd.read_csv(
  "./data/ml-100k/u.data", # tab으로 구분된 텍스트 데이터 파일
  sep="\t",
  names=["user_id", "item_id", "rating", "timestamp"]
) # 사용자, 아이템(영화), 평점, 시간 을 가져와줍니다.

print(type(ratings))
print(ratings.shape)
ratings.head()

<class 'pandas.DataFrame'>
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
(100000, 4)


In [6]:
print(ratings.dtypes)

user_id      int64
item_id      int64
rating       int64
timestamp    int64
dtype: object


In [9]:
print("user_id:", ratings["user_id"].min(), ratings["user_id"].max())
print("item_id (movie):", ratings["item_id"].min(), ratings["item_id"].max())

user_id: 1 943
item_id (movie): 1 1682


In [13]:
ratings["user_idx"] = ratings["user_id"] - 1
ratings["item_idx"] = ratings["item_id"] - 1

In [14]:
positive_ratings = ratings[
  ratings["rating"] >= 4 # bool마스크를 이용해서 선택 인덱스(4점 이상)을 새로운 df로 추출
].copy()

In [15]:
print(
  positive_ratings[
    ["user_idx", "item_idx", "rating"]
  ]
)

print(len(positive_ratings))

       user_idx  item_idx  rating
5           297       473       4
7           252       464       5
11          285      1013       5
12          199       221       5
16          121       386       5
...         ...       ...     ...
99988       420       497       4
99989       494      1090       4
99990       805       420       4
99991       675       537       4
99996       715       203       5

[55375 rows x 3 columns]
55375


In [19]:
# 사용자별 시계열 split를 하기 위해 sort 시켜주기
positive_ratings = positive_ratings.sort_values(["user_idx", "timestamp"])
positive_ratings[["user_idx", "item_idx", "rating", "timestamp"]].head(10)

,user_idx,item_idx,rating,timestamp
59972,0,167,5,874965478
92487,0,171,5,874965478
74577,0,164,5,874965518
48214,0,155,4,874965556
15764,0,195,5,874965677
22971,0,165,5,874965677
21626,0,186,4,874965678
9170,0,13,5,874965706
12948,0,249,4,874965706
53552,0,126,5,874965706


In [28]:
positive_ratings["user_idx"].nunique()

942

In [24]:
counts = positive_ratings.groupby("user_idx").size()
print(counts, counts.mean(), counts.std())

user_idx
0      163
1       40
2       15
3       19
4       58
      ... 
938     39
939     57
940     18
941     66
942     96
Length: 942, dtype: int64 58.78450106157113 54.69666379694414


In [29]:
# 2개 이상의 긍정적인 평가를 남긴 사용자들의 인덱스들만 뽑기
valid_users = counts[counts >= 2].index
valid_users.nunique()

942

In [31]:
positive_ratings["user_idx"].isin(valid_users).dtype

dtype('bool')

In [ ]:
positive_ratings = positive_ratings[
  # bool mask 넣어서 뽑아주기
  positive_ratings["user_idx"].isin(valid_users)
].copy()

In [32]:
type(positive_ratings)

pandas.DataFrame

In [35]:
# 각 사용자별 마지막 평점을 테스트로 뽑기 (나중에 선택한 [사용자, 영화] 내적이 선호하는 형태인지 확인하기 위함)
test_positive = (
  positive_ratings
  .groupby("user_idx")
  .tail(1)
  .drop(columns=["user_id", "item_id"])
)

In [37]:
test_positive.head(10)

,rating,timestamp,user_idx,item_idx
30479,4,889751712,0,255
12150,5,888979693,1,315
37188,4,889237482,2,180
48826,4,892004520,3,10
2686,4,879198229,4,23
38237,4,883717304,5,271
87329,5,892135347,6,356
44635,4,879362423,7,226
30909,5,886960056,8,482
2291,4,880371312,9,339


In [38]:
test_positive.index

Index([30479, 12150, 37188, 48826,  2686, 38237, 87329, 44635, 30909,  2291,
       ...
       89365, 68187, 98467, 89979, 91072, 80434, 82942, 98516, 88898, 86663],
      dtype='int64', length=942)

In [52]:
train_positive = positive_ratings.drop(
  test_positive.index
).drop(columns=["user_id", "item_id"])

In [53]:
user_seen_items = (
  ratings
  .groupby("user_idx")["item_idx"] # 사용자가 선택한 모든 item_idx를 선택
  .apply(set) # 사용자가 선택한 item_idx를 set으로 바꾸기
  .to_dict() # 딕셔너리 형태로 변형하기
)
[(user_id, len(items)) for user_id, items in user_seen_items.items()][0]

(0, 272)

In [54]:
import random

all_items = set(ratings["item_idx"].unique())

train_users = []
train_items = []
train_labels = []

num_negatives = 3

In [55]:
next(train_positive.itertuples())

Pandas(Index=59972, rating=5, timestamp=874965478, user_idx=0, item_idx=167)

In [ ]:
type(test_positive), test_positive.columns

(pandas.DataFrame,
 Index(['rating', 'timestamp', 'user_idx', 'item_idx'], dtype='str'))

In [ ]:
type(all_items), type(user_seen_items)

(set, dict)

In [57]:
for row in train_positive.itertuples():
  user_idx = row.user_idx
  positive_item = row.item_idx

  # positive sample
  train_users.append(user_idx)
  train_items.append(positive_item)
  train_labels.append(row.rating)

  # 사용자가 한 번도 평가하지 않은 아이템
  negative_candidates = list(
    # 모든 영화중에서 사용자가 본 아이템 제외하기
    all_items - user_seen_items[user_idx]
  )

  # 사용자가 본적 없는 영화를 섞어서 k개 뽑기
  sampled_negatives = random.sample(
    negative_candidates,
    k=num_negatives
  )

  # user_idx 한명이 하나의 긍정적인 평가를 했다면 num_negatives개의 부정적인 평가를 학습 데이터에 추가해주기
  for negative_item in sampled_negatives:
    train_users.append(user_idx)
    train_items.append(negative_item)
    train_labels.append(0.0)

In [59]:
import torch

# 임베딩 인덱스로 사용할 예정 -> 인덱스로 쓰기 위해 정수(long)
train_users = torch.tensor(train_users, dtype=torch.long)
train_items = torch.tensor(train_items, dtype=torch.long)
# 모델이 산출한 점수를 시그모이드에 넣은 값과 비교해야하는 값 -> float32
train_labels = torch.tensor(train_labels, dtype=torch.float32)

C:\Users\hurwa\AppData\Local\Temp\ipykernel_2776\1814502604.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_users = torch.tensor(train_users, dtype=torch.long)
C:\Users\hurwa\AppData\Local\Temp\ipykernel_2776\1814502604.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_items = torch.tensor(train_items, dtype=torch.long)
C:\Users\hurwa\AppData\Local\Temp\ipykernel_2776\1814502604.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.float32)


In [63]:
import torch.nn as nn

class Recommender(nn.Module):
  def __init__(self, num_users, num_items, embedding_dim=32):
    super().__init__()

    self.user_embedding = nn.Embedding(num_users, embedding_dim=embedding_dim)
    self.item_embedding = nn.Embedding(num_items, embedding_dim=embedding_dim)

  def forward(self, user_ids, item_ids):
    users_vec = self.user_embedding(user_ids)
    items_vec = self.item_embedding(item_ids)

    return (users_vec * items_vec).sum(dim=1)


In [66]:
model = Recommender(
  num_users=ratings["user_idx"].nunique(),
  num_items=ratings["item_idx"].nunique(),
  embedding_dim=32
)

# sigmoid + 대소비교
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
  model.parameters(),
  lr=0.001
)

In [72]:
ratings["user_idx"].nunique(), \
ratings["item_idx"].nunique(), \
sum([p.numel() for n, p in model.named_parameters()]), \
(ratings["user_idx"].nunique() + ratings["item_idx"].nunique()) * 32

(943, 1682, 84000, 84000)

In [ ]:
once_flag=False

for epoch in range(10):
  optimizer.zero_grad()

  # 두 shape=(1,)의 LongTensor 리스트에 대해서 유사도 리스트(float32)로 변형해주기
  # 다르게 이야기하면 사용자-영화의 유사도를 뽑아주기. (n, 32) 형태로
  logits = model(
    train_users,
    train_items
  )

  if once_flag == False:
    print(logits.dtype)
    print(len(logits))
    once_flag=True

  loss = criterion(
    logits,
    train_labels
  )

  loss.backward()
  optimizer.step()

  print(f"epoch: {epoch} | loss:{loss.item()}")

torch.float32
217732
epoch: 0 | loss:2.3271520137786865
epoch: 1 | loss:2.3194079399108887
epoch: 2 | loss:2.311673402786255
epoch: 3 | loss:2.303947687149048
epoch: 4 | loss:2.2962307929992676
epoch: 5 | loss:2.2885239124298096
epoch: 6 | loss:2.2808258533477783
epoch: 7 | loss:2.273137331008911
epoch: 8 | loss:2.26545786857605
epoch: 9 | loss:2.2577877044677734


In [73]:
for epoch in range(1, 101):
  optimizer.zero_grad()

  # 두 shape=(1,)의 LongTensor 리스트에 대해서 유사도 리스트(float32)로 변형해주기
  # 다르게 이야기하면 사용자-영화의 유사도를 뽑아주기. (n, 32) 형태로
  logits = model(
    train_users,
    train_items
  )

  if once_flag == False:
    print(logits.dtype)
    print(len(logits))
    once_flag=True

  loss = criterion(
    logits,
    train_labels
  )

  loss.backward()
  optimizer.step()

  if epoch % 50 == 0:
    print(f"epoch: {epoch} | loss:{loss.item()}")

epoch: 50 | loss:1.9923354387283325
epoch: 100 | loss:1.6125035285949707


In [75]:
user_idx = 0

test_positive[test_positive["user_idx"] == user_idx]

,rating,timestamp,user_idx,item_idx
30479,4,889751712,0,255


In [86]:
items_df =  pd.read_csv(
  "./data/ml-100k/u.item",
  sep="|",
  encoding="latin-1",
  header=None,
  usecols=[0, 1],
  names=["item_id", "title"]
)

items_df["item_idx"] = items_df["item_id"] - 1
items_df.head()

,item_id,title,item_idx
0,1,Toy Story (1995),0
1,2,GoldenEye (1995),1
2,3,Four Rooms (1995),2
3,4,Get Shorty (1995),3
4,5,Copycat (1995),4


In [87]:
user_idx = 0

num_items = ratings["item_idx"].nunique()

all_item_tensor = torch.arange(num_items)

user_tensor = torch.full(
  (num_items,),
  user_idx,
  dtype=torch.long
)

In [88]:
num_items

1682

In [ ]:
model.eval()

with torch.no_grad():
  scores = model(
    user_tensor,
    all_item_tensor
  )

  seen_items = list(user_seen_items[user_idx])

  # 이미 본 아이템들에 대해서 모두 추천도 최악으로 만들고 새로운 아이템 보여주기
  scores[seen_items] = -float("inf")

  top_scores, top_items = torch.topk(
    scores,
    k=10
  )

  # recommendations = items_bdf[
  #   items_df["item_idx"].isin(top_items.tolist())
  # ][["item_idx", "title"]]

In [89]:
item_to_title = dict(
  zip(
    items_df["item_idx"],
    items_df["title"]
  )
)

In [85]:
test_positive[test_positive["user_idx"] == user_idx]

,rating,timestamp,user_idx,item_idx
30479,4,889751712,0,255


In [94]:
item_to_title[top_items[0].item()]

'Gandhi (1982)'

In [95]:
result = pd.DataFrame(
  data={
    "top_item": [item_to_title[idx.item()] for idx in top_items],
    "score": top_scores, 
  },
  index=range(1, 11, 1)
)
result.T

,1,2,3,4,5,6,7,8,9,10
top_item,Gandhi (1982),Blue Chips (1994),Up in Smoke (1978),Up Close and Personal (1996),"Secret Garden, The (1993)",Winnie the Pooh and the Blustery Day (1968),Heavenly Creatures (1994),Dial M for Murder (1954),Paris Was a Woman (1995),"Replacement Killers, The (1998)"
score,16.368538,16.127693,15.820114,15.735778,15.54023,15.016037,14.632208,14.604223,14.188915,13.943156
